# Hearing Risk — Colab runner

Run the cells top to bottom. **Do not edit code here**; all code lives in the repo.
The only cell you change is the one marked **PASTE THE TASK COMMAND HERE** (and `BRANCH` / the copy flags below).

Before running: Runtime → Change runtime type → **CPU** for data tasks, **T4 GPU** for training.

## 1. Settings

In [ ]:
BRANCH = "main"          # branch to run; leave as main after the PR is merged
REPO_URL = "https://github.com/ethany18/APS360Project-HearingRisk.git"
REPO_DIR = "/content/APS360Project-HearingRisk"

DRIVE_ROOT = "/content/drive/MyDrive/HearingRisk"   # raw/, processed/, checkpoints/, results/
LOCAL_DATA = "/content/data"                         # fast local disk; lost when the runtime ends

COPY_MDRM_RAW = True     # merge + unzip MDRM audio onto local disk (needed for Tasks 1, 4, 6)
COPY_PROCESSED = False   # copy Drive processed/ to local disk (turn on once processed/ exists)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
for sub in ["processed", "checkpoints", "results"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
print(sorted(os.listdir(DRIVE_ROOT)))

## 3. Clone or pull the repo (branch = `BRANCH`)

In [ ]:
import os, subprocess
if not os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
!git log --oneline -3

## 4. Install dependencies

In [ ]:
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 5. Copy data from Drive to `/content/data`
Reading many small files from Drive is very slow, so everything is copied to local disk first.
MDRM: the split zip on Drive is merged and unzipped to `/content/data/mdrm` (skipped if already there).
FinCall-Surprise (52 GB) is **not** copied; only checked that the shortcut is visible.

In [ ]:
%cd {REPO_DIR}
import os
os.makedirs(LOCAL_DATA, exist_ok=True)
if COPY_MDRM_RAW:
    !python -m src.data.unpack_mdrm --src "{DRIVE_ROOT}/raw/mdrm" --dst "{LOCAL_DATA}/mdrm"
if COPY_PROCESSED:
    !mkdir -p "{LOCAL_DATA}/processed" && rsync -a --info=stats1 "{DRIVE_ROOT}/processed/" "{LOCAL_DATA}/processed/"
fc = f"{DRIVE_ROOT}/raw/FinCall-Surprise"
print("FinCall-Surprise shortcut:", sorted(os.listdir(fc)) if os.path.isdir(fc) else "NOT FOUND")
!df -h /content | tail -1

## 6. ▶▶▶ PASTE THE TASK COMMAND HERE ◀◀◀
Replace the contents of the cell below with the command Claude Code gave you for the current task.
(Pre-filled with Task 1: inventory of MDRM.)

In [ ]:
%cd {REPO_DIR}
# ===== PASTE THE TASK COMMAND BELOW THIS LINE =====
!python -m src.data.inventory --root /content/data/mdrm --out-dir results --copy-to "{DRIVE_ROOT}/processed"
# ===== END OF TASK COMMAND =====

## 7. Copy `results/` back to Drive
Copies to `HearingRisk/results/` and downloads a zip, so you can upload the files to GitHub
(Add file → Upload files, into `results/` on `main`). Run this before closing the tab.

In [ ]:
%cd {REPO_DIR}
!mkdir -p "{DRIVE_ROOT}/results" && rsync -a results/ "{DRIVE_ROOT}/results/"
!ls -la "{DRIVE_ROOT}/results"
!zip -q -r /content/results.zip results
from google.colab import files
files.download("/content/results.zip")